In [1]:
import pandas as pd

In [8]:
# Load the raw files
transactions = pd.read_csv('Retail_Data_Transactions.csv')
response = pd.read_csv('Retail_Data_Response.csv')

In [7]:
# Load the raw files
transactions = pd.read_csv('Retail_Data_Transactions.csv')

In [9]:
# Look at the first 5 rows of transactions
print("--- Transactions Sample ---")
print(transactions.head())

--- Transactions Sample ---
  customer_id trans_date  tran_amount
0      CS5295  11-Feb-13           35
1      CS4768  15-Mar-15           39
2      CS2122  26-Feb-13           52
3      CS1217  16-Nov-11           99
4      CS1850  20-Nov-13           78


In [10]:
# Look at the first 5 rows of response
print("\n--- Response Sample ---")
print(response.head())


--- Response Sample ---
  customer_id  response
0      CS1112         0
1      CS1113         0
2      CS1114         1
3      CS1115         1
4      CS1116         1


In [11]:
# Check data types and summary info
print(transactions.info())
print(response.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125000 entries, 0 to 124999
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   customer_id  125000 non-null  object
 1   trans_date   125000 non-null  object
 2   tran_amount  125000 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6884 entries, 0 to 6883
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  6884 non-null   object
 1   response     6884 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 107.7+ KB
None


In [13]:
# Convert the date column from text to a proper datetime format
transactions['trans_date'] = pd.to_datetime(transactions['trans_date'])

In [14]:
# Verify it worked
print(transactions['trans_date'].dtype)

datetime64[ns]


In [15]:
import numpy as np

# 1. Convert trans_date column to proper datetime format (just to be safe)
transactions['trans_date'] = pd.to_datetime(transactions['trans_date'])

# 2. Find the most recent date in the entire dataset to act as our benchmark
latest_date = transactions['trans_date'].max()
print(f"The latest transaction date in dataset is: {latest_date.strftime('%Y-%m-%d')}")

# 3. Aggregate data by customer_id to compute Recency, Frequency, and Monetary values
rfm_df = transactions.groupby('customer_id').agg({
    # Recency: How many days between the latest date and the customer's last purchase
    'trans_date': lambda x: (latest_date - x.max()).days,
    # Frequency: Total number of transactions made by the customer
    'customer_id': 'count',
    # Monetary: Sum of all transaction amounts for the customer
    'tran_amount': 'sum'
})

# 4. Rename the columns so they are clean and readable
rfm_df.rename(columns={
    'trans_date': 'recency',
    'customer_id': 'frequency',
    'tran_amount': 'monetary'
}, inplace=True)

# Reset index to bring 'customer_id' back as a normal column
rfm_df = rfm_df.reset_index()

# 5. Merge the RFM metrics with the Response table on customer_id
final_master_df = rfm_df.merge(response, on='customer_id', how='left')

# 6. Fill any missing response values with 0 (in case a customer had no response record)
final_master_df['response'] = final_master_df['response'].fillna(0).astype(int)

# 7. Print a preview of your newly created master dataset
print("\n--- Master RFM Dataset Preview ---")
print(final_master_df.head())

The latest transaction date in dataset is: 2015-03-16

--- Master RFM Dataset Preview ---
  customer_id  recency  frequency  monetary  response
0      CS1112       61         15      1012         0
1      CS1113       35         20      1490         0
2      CS1114       32         19      1432         1
3      CS1115       11         22      1659         1
4      CS1116      203         13       857         1


In [16]:
# Save the master dataset as a CSV file in your Colab environment
final_master_df.to_csv('Processed_Retail_Data.csv', index=False)
print("File 'Processed_Retail_Data.csv' successfully created!")

File 'Processed_Retail_Data.csv' successfully created!


In [17]:
!pip install SQLAlchemy pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.8 MB/s eta 0:00:00


In [25]:
# Save the files directly as CSVs to download
transactions.to_csv('clean_transactions.csv', index=False)
response.to_csv('clean_response.csv', index=False)
print("Files are ready for download!")

Files are ready for download!
